# OMOP to TTE: Native Terra Execution Pipeline (V1.5)

This notebook demonstrates how to load the `omop_tte_bridge` module directly within a secure cloud Jupyter workspace, run schema learning using Vertex AI, and generate BigQuery SQL.

In [ ]:
!pip install -e ../

In [ ]:
from omop_tte_bridge.discovery import schema_learner
from omop_tte_bridge.queries import query_generator

# 1. Run Schema Learning (Extracts INFORMATION_SCHEMA and summons Vertex AI)
# This step detects which specific biobank dialect we are currently operating within.
topology_map = schema_learner.discover_schema(project_id="your-gcp-project", dataset_id="omop_v5")

print("Detected Biobank Dialect:", topology_map['dialect_detected'])


In [ ]:
# 2. Generate Exploratory Query
prompt = "Find the top 5 most common conditions for patients exposed to Metformin"
exploratory_sql = query_generator.build(topology_map, prompt, mode="exploratory")

print(exploratory_sql)

In [ ]:
# 3. Execute the Query via BigQuery Magic directly in Terra!
from google.cloud import bigquery
client = bigquery.Client()

df_exploratory = client.query(exploratory_sql).to_dataframe()
df_exploratory.head()

In [ ]:
# 4. Generate Final TTE Matrix Query
final_sql = query_generator.build(topology_map, prompt, mode="final")

df_final_matrix = client.query(final_sql).to_dataframe()

# 5. Save the data out for local Antigravity TTE analysis
df_final_matrix.to_csv("trial_cohort.csv", index=False)
df_final_matrix.to_json("trial_cohort.json", orient="records")
print("Now download the .json or .csv to your personal computer!")